<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-09/NB09_limits_of_machines.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 09: Limits of Machines: Gödelian Arguments and Dreyfus's Critique

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-09/lab.html>.

## Overview

Two very different arguments claim that minds can do something that machines cannot. One appeals to Gödel's incompleteness theorems, the other to the phenomenology of skilled human activity. This week reconstructs both, examines the replies and asks which limits of machines are real [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to state the incompleteness theorem and the undecidability of the halting problem informally, to construct a diagonal argument, to reconstruct the Lucas-Penrose argument and the consistency objection, and to explain Dreyfus's critique of rule-based AI and its relation to later developments.

## Incompleteness and undecidability

Gödel proved in 1931 that any consistent formal system that can express elementary arithmetic contains statements that are true but cannot be proved within the system [1]. The construction uses self-reference: a sentence that, in effect, says of itself that it is not provable. If the system is consistent, the sentence is unprovable and therefore true. Turing reached a related result for machines [4]. No Turing machine can decide, for every program and input, whether the program halts. The proof is a diagonal argument: Given any claimed decider, one can build a program that asks the decider about itself and does the opposite. Figure 9.1 shows the idea as a table in which the flipped diagonal differs from every row.

![Figure 9.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-09/figures/w09_fig1.png)

*Figure 9.1. A diagonal argument: flipping the diagonal of any table of programs and inputs yields a behaviour that differs from every program in the table [4].*

## The Lucas-Penrose argument

Lucas argued that the incompleteness theorem shows that minds are not machines [2]. For any machine that proves arithmetic, there is a Gödel sentence that the machine cannot prove, but a human mathematician can see that the sentence is true. So the mathematician can do something that the machine cannot, and no machine can be an adequate model of the mind. Penrose developed a version of the argument and concluded that human mathematical insight is not computable, which led him to speculate about new physics in the brain [5, 6].

The central objection concerns consistency. A human sees that the Gödel sentence of a machine is true only if the human knows that the machine is consistent. Benacerraf argued that the argument establishes at most a disjunction: Either the mind is not a machine, or it is a machine whose program it cannot know or whose consistency it cannot prove [7]. Since people are not known to be consistent, and cannot prove their own consistency, the argument does not reach its conclusion. Most logicians and philosophers regard the Gödelian argument as unsuccessful, though it continues to be refined and debated.

## Dreyfus's phenomenological critique

Dreyfus argued that the programme of symbolic AI rested on assumptions that were false [3, 8]. It assumed that the brain processes information in discrete operations, that the mind follows rules over symbols, that all knowledge can be formalised and that the world consists of independent facts. Drawing on Heidegger and Merleau-Ponty, he argued that human expertise is not rule-following. Experts respond directly to situations that show up as already meaningful against a background of practices, and this know-how cannot be captured in explicit rules. The frame problem of Week 8 was, for Dreyfus, a symptom of this mistake.

Later, Dreyfus examined attempts to build Heideggerian AI, including behaviour-based robotics, and argued that they had not gone far enough [9]. Brooks's robots, in his view, responded to fixed features of the world rather than to changing significance, and a truly Heideggerian AI would need a body with needs that makes situations matter to it.

## Which limits are real?

The two arguments differ in kind. The Gödelian argument claims an in-principle limit on any machine. It fails, according to most critics, because the same limits apply to humans. Dreyfus's critique targeted a particular approach, symbolic AI, and much of it has been vindicated by the shift to learning systems that acquire skills from experience rather than from explicit rules. What remains open is whether learned systems escape his deeper point about significance and embodiment. The undecidability of halting remains a genuine limit, but it binds every computing system, including brains, if they compute.

> **Pause and reflect.** The lab shows that a bounded halting checker is always wrong about some programs. Are you, as a human reasoner, subject to the same limit?

# Hands-on lab

The notebook builds a table of programs and inputs from simple iterated maps, constructs the diagonal program, evaluates bounded halting checkers and detects cycles exactly where possible [1, 4].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Programs as iterated maps

Program i applies a map to its input until it reaches 1. Halting is decided exactly for these cases by detecting cycles, and a run that neither halts nor cycles within 20,000 steps is marked as unknown.

In [ ]:
MAPS = {
    "collatz 3x+1": lambda x: x // 2 if x % 2 == 0 else 3 * x + 1,
    "5x+1": lambda x: x // 2 if x % 2 == 0 else 5 * x + 1,
    "halve or add 3": lambda x: x // 2 if x % 2 == 0 else x + 3,
    "halve or add 1": lambda x: x // 2 if x % 2 == 0 else x + 1,
    "double mod 11": lambda x: (2 * x) % 11,
    "square mod 13": lambda x: (x * x) % 13,
}
INPUTS = [2, 3, 5, 6, 7, 13]

def behaviour(f, x, limit=20_000):
    seen = set()
    for step in range(limit):
        if x == 1:
            return step
        if x in seen:
            return np.inf
        seen.add(x)
        x = f(x)
    return None

import pandas as pd
TABLE = pd.DataFrame({x: {name: behaviour(f, x) for name, f in MAPS.items()} for x in INPUTS})
HALTS = TABLE.apply(lambda col: col.map(lambda v: v is not None and bool(np.isfinite(v))))
HALTS

### Your turn, exercise 1

Treat the six maps as programs 0 to 5 and the six inputs as inputs 0 to 5. Build the diagonal program D as a list of six booleans, where D halts on input i exactly when program i does not halt on input i.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _diagonal_reference():
    return [not bool(HALTS.iloc[i, i]) for i in range(6)]

In [ ]:
D = None  # a list of six booleans
check("Exercise 1", D, _diagonal_reference())
if D is not None:
    print("D differs from every row:", all(D[i] != bool(HALTS.iloc[i, i]) for i in range(6)))

## 2. Bounded checkers

A bounded checker simulates for k steps and answers halts if the program stopped, otherwise loops.

In [ ]:
rng = np.random.default_rng(SEED)
sample = [(name, int(x)) for name in MAPS for x in rng.integers(2, 400, 40)]
truths = [behaviour(MAPS[n], x) for n, x in sample]
known = [(s, t) for s, t in zip(sample, truths) if t is not None]
print(len(known), "programs with known behaviour out of", len(sample))

def bounded_checker(name, x, k):
    t = behaviour(MAPS[name], x, limit=k)
    return t is not None and np.isfinite(t)

### Your turn, exercise 2

Compute the accuracy of the bounded checker with k = 50 on the programs with known behaviour. Then plot the accuracy for k from 1 to 200 and explain why it never guarantees correctness for all programs.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _bounded_accuracy_reference(k=50):
    return float(np.mean([bounded_checker(n, x, k) == bool(np.isfinite(t)) for (n, x), t in known]))

In [ ]:
bounded_accuracy = None  # your computation
check("Exercise 2", bounded_accuracy, _bounded_accuracy_reference())
ks = range(1, 201, 5)
plt.plot(list(ks), [_bounded_accuracy_reference(k) for k in ks]); plt.xlabel("budget k"); plt.ylabel("accuracy"); plt.title("A bounded halting checker"); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-09/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Does undecidability limit human reasoners as well as machines? Explain how you would argue either way.
2. Which parts of Dreyfus's critique were vindicated by machine learning, and which remain challenges for present systems?
3. Choose a skill you possess that you could not reduce to rules. What does that suggest about the systems you build?

## Weekly task and submission

Write about 500 words reconstructing the Lucas-Penrose argument, presenting the consistency objection and assessing whether any version of the argument survives. Use the diagonal construction from the notebook in your explanation. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Gödelian arguments today.** Review contemporary defences and critiques of the Lucas-Penrose argument and assess whether any version survives the consistency objection [2, 6, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Gödel, K. (1931). Über formal unentscheidbare Sätze der Principia Mathematica und verwandter Systeme I. *Monatshefte für Mathematik und Physik*, 38(1), 173-198. <https://doi.org/10.1007/BF01700692>

[2] Lucas, J. R. (1961). Minds, machines and Gödel. *Philosophy*, 36(137), 112-127. <https://doi.org/10.1017/S0031819100057983>

[3] Dreyfus, H. L. (1992). *What Computers Still Can't Do: A Critique of Artificial Reason*. MIT Press.

[4] Turing, A. M. (1936). On computable numbers, with an application to the Entscheidungsproblem. *Proceedings of the London Mathematical Society*, s2-42(1), 230-265. <https://doi.org/10.1112/plms/s2-42.1.230>

[5] Penrose, R. (1989). *The Emperor's New Mind: Concerning Computers, Minds, and the Laws of Physics*. Oxford University Press.

[6] Penrose, R. (1994). *Shadows of the Mind: A Search for the Missing Science of Consciousness*. Oxford University Press.

[7] Benacerraf, P. (1967). God, the devil, and Gödel. *The Monist*, 51(1), 9-32. <https://doi.org/10.5840/monist196751112>

[8] Dreyfus, H. L. (1972). *What Computers Can't Do: A Critique of Artificial Reason*. Harper & Row.

[9] Dreyfus, H. L. (2007). Why Heideggerian AI failed and how fixing it would require making it more Heideggerian. *Philosophical Psychology*, 20(2), 247-268. <https://doi.org/10.1080/09515080701239510>